In [2]:
# ======================================
# 0. Imports
# ======================================
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from scipy.sparse import hstack, csr_matrix

np.random.seed(42)

# ======================================
# 1. Load data from dataset folder
# ======================================
movies = pd.read_csv("dataset/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv("dataset/ratings.csv")

# ---- clean movie ids ----
movies = movies[movies["id"].str.isnumeric()]
movies["id"] = movies["id"].astype(int)

# ---- drop duplicates ----
movies = movies.drop_duplicates(subset="id").reset_index(drop=True)
ratings = ratings[ratings["movieId"].isin(movies["id"])]

# ======================================
# 2. Movie features
# ======================================
def parse_genres(x):
    try:
        return [g["name"] for g in eval(x)]
    except:
        return []

movies["genres_list"] = movies["genres"].apply(parse_genres)
movies["overview"] = movies["overview"].fillna("")
movies["tagline"] = movies["tagline"].fillna("")
movies["text"] = movies["overview"] + " " + movies["tagline"]

# ---- Text features ----
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2))
X_text = tfidf.fit_transform(movies["text"])

# ---- Genres ----
mlb = MultiLabelBinarizer()
X_genres = csr_matrix(mlb.fit_transform(movies["genres_list"]))

# ---- Numeric ----
num_cols = ["popularity", "runtime", "vote_average", "vote_count"]
movies[num_cols] = movies[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
scaler = StandardScaler()
X_num = csr_matrix(scaler.fit_transform(movies[num_cols]))

# ---- Item feature space ----
X_item_raw = hstack([X_text, X_genres, X_num])
print("Raw item feature space:", X_item_raw.shape)

# ======================================
# 3. Item embeddings
# ======================================
svd = TruncatedSVD(n_components=128, random_state=42)
X_item = svd.fit_transform(X_item_raw)
print("Item embedding space:", X_item.shape)

# ---- Safe mapping ----
movie_id_to_idx = dict(zip(movies["id"], range(len(movies))))

# ======================================
# 4. Ratings → indices
# ======================================
ratings = ratings.copy()
ratings["movie_idx"] = ratings["movieId"].map(movie_id_to_idx)
ratings = ratings.dropna(subset=["movie_idx"])
ratings["movie_idx"] = ratings["movie_idx"].astype(int)

# ======================================
# 5. User embeddings (weighted average of rated movies)
# ======================================
user_embeddings = {}
for user_id, grp in ratings.groupby("userId"):
    idxs = grp["movie_idx"].values
    weights = grp["rating"].values
    user_embeddings[user_id] = np.average(X_item[idxs], axis=0, weights=weights)

print("Number of users with embeddings:", len(user_embeddings))

# ======================================
# 6. Train classifier in batches (memory-efficient)
# ======================================
batch_size = 50000
model = SGDClassifier(loss="log_loss", max_iter=1, learning_rate="optimal")

ratings_list = list(ratings.itertuples(index=False))
n_batches = int(np.ceil(len(ratings_list) / batch_size))

for epoch in range(5):  # несколько эпох
    for i in range(n_batches):
        batch = ratings_list[i*batch_size:(i+1)*batch_size]
        X_batch, y_batch = [], []
        for row in batch:
            u, m = row.userId, row.movie_idx
            if u not in user_embeddings:
                continue
            X_batch.append(np.hstack([user_embeddings[u], X_item[m]]))
            y_batch.append(1 if row.rating >= 4.0 else 0)
        if X_batch:
            X_batch = np.array(X_batch, dtype=np.float32)
            y_batch = np.array(y_batch)
            if epoch == 0 and i == 0:
                model.partial_fit(X_batch, y_batch, classes=[0,1])
            else:
                model.partial_fit(X_batch, y_batch)
    print(f"Epoch {epoch+1} done")

# ======================================
# 7. Recommendation function
# ======================================
def recommend_for_user(user_id, top_n=10):
    if user_id not in user_embeddings:
        return None
    user_vec = user_embeddings[user_id].reshape(1,-1)
    user_block = np.repeat(user_vec, X_item.shape[0], axis=0)
    X_pred = np.hstack([user_block, X_item])
    scores = model.predict_proba(X_pred)[:,1]
    
    seen = ratings[ratings["userId"] == user_id]["movieId"].values
    recs = movies.copy()
    recs["score"] = scores
    recs = recs[~recs["id"].isin(seen)]
    return recs.sort_values("score", ascending=False)[["title", "score", "genres_list"]].head(top_n)

# ======================================
# 8. Example
# ======================================
print("\nRecommendations for user 1:")
print(recommend_for_user(1))


Raw item feature space: (45433, 5024)
Item embedding space: (45433, 128)
Number of users with embeddings: 265917
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done

Recommendations for user 1:
                                              title     score  \
30677                                       Minions  0.840955   
16481                               The Fern Flower  0.820101   
39650                            Komunaris chibukhi  0.801518   
35903                               Poil de Carotte  0.774752   
3529                        City of the Living Dead  0.768445   
40297                                A Simple Story  0.761717   
42164               Mia trelli... trelli oikogeneia  0.760367   
18198             Extremely Loud & Incredibly Close  0.759424   
2097                                    Jamaica Inn  0.749888   
21191  Return of the Living Dead: Rave to the Grave  0.749526   

                                  genres_list  
30677  [Family, Animation, Ad

In [1]:
# ======================================
# 0. Imports
# ======================================
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import hstack, csr_matrix

np.random.seed(42)

# ======================================
# 1. Load data
# ======================================
movies = pd.read_csv("dataset/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv("dataset/ratings.csv")

movies = movies[movies["id"].str.isnumeric()]
movies["id"] = movies["id"].astype(int)

movies = movies.drop_duplicates(subset="id").reset_index(drop=True)
ratings = ratings[ratings["movieId"].isin(movies["id"])]

# ======================================
# 2. Movie features
# ======================================
def parse_genres(x):
    try:
        return [g["name"] for g in eval(x)]
    except:
        return []

movies["genres_list"] = movies["genres"].apply(parse_genres)
movies["overview"] = movies["overview"].fillna("")
movies["tagline"] = movies["tagline"].fillna("")
movies["text"] = movies["overview"] + " " + movies["tagline"]

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2)
)
X_text = tfidf.fit_transform(movies["text"])

mlb = MultiLabelBinarizer()
X_genres = csr_matrix(mlb.fit_transform(movies["genres_list"]))

num_cols = ["popularity", "runtime", "vote_average", "vote_count"]
movies[num_cols] = movies[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

scaler = StandardScaler()
X_num = csr_matrix(scaler.fit_transform(movies[num_cols]))

X_item_raw = hstack([X_text, X_genres, X_num])
print("Raw item features:", X_item_raw.shape)

# ======================================
# 3. Item embeddings
# ======================================
svd = TruncatedSVD(n_components=128, random_state=42)
X_item = svd.fit_transform(X_item_raw)
print("Item embeddings:", X_item.shape)

movie_id_to_idx = dict(zip(movies["id"], range(len(movies))))

# ======================================
# 4. Ratings → indices
# ======================================
ratings = ratings.copy()
ratings["movie_idx"] = ratings["movieId"].map(movie_id_to_idx)
ratings = ratings.dropna(subset=["movie_idx"])
ratings["movie_idx"] = ratings["movie_idx"].astype(int)

# ======================================
# 5. User embeddings (weighted mean)
# ======================================
user_embeddings = {}
for user_id, grp in ratings.groupby("userId"):
    idxs = grp["movie_idx"].values
    weights = grp["rating"].values
    user_embeddings[user_id] = np.average(
        X_item[idxs], axis=0, weights=weights
    )

user_ids = np.array(list(user_embeddings.keys()))
X_user = np.vstack(list(user_embeddings.values()))
user_id_to_idx = dict(zip(user_ids, range(len(user_ids))))

print("Users with embeddings:", len(user_embeddings))

# ======================================
# 6. Train user-item classifier
# ======================================
batch_size = 50000
model = SGDClassifier(
    loss="log_loss",
    max_iter=1,
    learning_rate="optimal"
)

ratings_list = list(ratings.itertuples(index=False))
n_batches = int(np.ceil(len(ratings_list) / batch_size))

for epoch in range(5):
    for i in range(n_batches):
        batch = ratings_list[i*batch_size:(i+1)*batch_size]

        X_batch, y_batch = [], []
        for row in batch:
            u, m = row.userId, row.movie_idx
            if u not in user_embeddings:
                continue

            X_batch.append(
                np.hstack([user_embeddings[u], X_item[m]])
            )
            y_batch.append(1 if row.rating >= 4.0 else 0)

        if X_batch:
            X_batch = np.array(X_batch, dtype=np.float32)
            y_batch = np.array(y_batch)

            if epoch == 0 and i == 0:
                model.partial_fit(X_batch, y_batch, classes=[0,1])
            else:
                model.partial_fit(X_batch, y_batch)

    print(f"Epoch {epoch+1} finished")

# ======================================
# 7. Recommendation logic (IMPROVED)
# ======================================
def recommend_for_user(user_id, top_n=10, alpha=0.7, beta=0.3):
    """
    alpha — classifier weight
    beta  — embedding similarity weight
    """
    if user_id not in user_embeddings:
        return None

    # --- classifier score ---
    user_vec = user_embeddings[user_id].reshape(1,-1)
    user_block = np.repeat(user_vec, X_item.shape[0], axis=0)
    X_pred = np.hstack([user_block, X_item])
    clf_score = model.predict_proba(X_pred)[:,1]

    # --- embedding similarity ---
    emb_score = cosine_similarity(
        user_vec, X_item
    )[0]

    # --- popularity penalty (long-tail boost) ---
    pop = movies["popularity"].values
    pop_penalty = 1 / (1 + np.log1p(pop))

    final_score = (
        alpha * clf_score +
        beta * emb_score
    ) * pop_penalty

    seen = ratings[ratings.userId == user_id]["movieId"].values
    recs = movies.copy()
    recs["score"] = final_score
    recs = recs[~recs["id"].isin(seen)]

    return recs.sort_values(
        "score", ascending=False
    )[["id","title","genres_list","score"]].head(top_n)


def similar_users(user_id, top_n=5):
    idx = user_id_to_idx[user_id]
    sims = cosine_similarity(
        X_user[idx].reshape(1,-1), X_user
    )[0]

    top_idx = np.argsort(sims)[::-1][1:top_n+1]
    return list(zip(user_ids[top_idx], sims[top_idx]))

# ======================================
# 8. LLM PROMPTS (IMPROVED)
# ======================================
def build_item_prompt(user_id):
    recs = recommend_for_user(user_id, top_n=5)
    sims = similar_users(user_id, top_n=3)

    prompt = f"""
You are an expert recommender system.

User ID: {user_id}

Similar users with close taste profiles:
"""

    for u,s in sims:
        prompt += f"- user {u}, similarity {s:.2f}\n"

    prompt += "\nCandidate movies:\n"
    for _, r in recs.iterrows():
        prompt += f"- {r['title']} | genres: {r['genres_list']}\n"

    prompt += """
Explain why these movies fit the user's preferences.
Focus on genre patterns, diversity, and latent taste dimensions.
Avoid generic explanations.
"""

    return prompt


def build_rerank_prompt(user_id, candidates):
    prompt = f"""
User {user_id} has the following candidate recommendations:
"""
    for _, r in candidates.iterrows():
        prompt += f"- {r['title']} ({r['genres_list']})\n"

    prompt += """
Re-rank these movies from best to worst
and provide a short justification for the top 3.
"""
    return prompt


def build_user_prompt(user_id):
    sims = similar_users(user_id, top_n=5)

    prompt = f"""
User ID: {user_id}

Most similar users:
"""
    for u, s in sims:
        prompt += f"- user {u}, similarity {s:.2f}\n"

    prompt += """
Explain what common preferences and viewing patterns
connect these users.
"""
    return prompt

# ======================================
# 9. Recommendations for 5 users
# ======================================

np.random.seed(42)

sample_users = np.random.choice(
    list(user_embeddings.keys()),
    size=5,
    replace=False
)

for uid in sample_users:
    print("\n" + "="*80)
    print(f"USER {uid}")
    print("="*80)

    recs = recommend_for_user(uid, top_n=5)

    print("\n--- RECOMMENDED MOVIES ---")
    print(recs[["title", "genres_list", "score"]])

    print("\n--- SIMILAR USERS ---")
    print(similar_users(uid, top_n=3))

    print("\n--- LLM ITEM PROMPT ---")
    print(build_item_prompt(uid))



Raw item features: (45433, 5024)
Item embeddings: (45433, 128)
Users with embeddings: 265917
Epoch 1 finished
Epoch 2 finished
Epoch 3 finished
Epoch 4 finished
Epoch 5 finished

USER 187762

--- RECOMMENDED MOVIES ---
                              title  \
35903               Poil de Carotte   
40297                A Simple Story   
19475                     The Crush   
27750                           Pan   
36827  The Clan - Tale of the Frogs   

                                        genres_list     score  
35903                               [Drama, Comedy]  0.525845  
40297                                            []  0.523881  
19475                              [Romance, Drama]  0.520985  
27750  [Action, Adventure, Fantasy, Drama, Romance]  0.511017  
36827       [Action, Drama, Crime, Comedy, Romance]  0.505723  

--- SIMILAR USERS ---
[(194761, 0.987813476500039), (1612, 0.9866846048276616), (96023, 0.9863816162455588)]

--- LLM ITEM PROMPT ---

You are an expert recommen

In [2]:
# ======================================
# 9. Recommendations for 5 users
# ======================================

np.random.seed(42)

sample_users = np.random.choice(
    list(user_embeddings.keys()),
    size=5,
    replace=False
)

for uid in sample_users:
    print("\n" + "="*80)
    print(f"USER {uid}")
    print("="*80)

    recs = recommend_for_user(uid, top_n=5)

    print("\n--- RECOMMENDED MOVIES ---")
    print(recs[["title", "genres_list", "score"]])

    print("\n--- SIMILAR USERS ---")
    print(similar_users(uid, top_n=3))

    print("\n--- LLM ITEM PROMPT ---")
    print(build_item_prompt(uid))



USER 187762

--- RECOMMENDED MOVIES ---
                              title  \
35903               Poil de Carotte   
40297                A Simple Story   
19475                     The Crush   
27750                           Pan   
36827  The Clan - Tale of the Frogs   

                                        genres_list     score  
35903                               [Drama, Comedy]  0.525845  
40297                                            []  0.523881  
19475                              [Romance, Drama]  0.520985  
27750  [Action, Adventure, Fantasy, Drama, Romance]  0.511017  
36827       [Action, Drama, Crime, Comedy, Romance]  0.505723  

--- SIMILAR USERS ---
[(194761, 0.987813476500039), (1612, 0.9866846048276616), (96023, 0.9863816162455588)]

--- LLM ITEM PROMPT ---

You are an expert recommender system.

User ID: 187762

Similar users with close taste profiles:
- user 194761, similarity 0.99
- user 1612, similarity 0.99
- user 96023, similarity 0.99

Candidate movies

In [3]:
# ======================================
# 0. Imports
# ======================================
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import hstack, csr_matrix

np.random.seed(42)

# ======================================
# 1. Load data
# ======================================
movies = pd.read_csv("dataset/movies_metadata.csv", low_memory=False)
ratings = pd.read_csv("dataset/ratings.csv")

movies = movies[movies["id"].str.isnumeric()]
movies["id"] = movies["id"].astype(int)

movies = movies.drop_duplicates(subset="id").reset_index(drop=True)
ratings = ratings[ratings["movieId"].isin(movies["id"])]

# ======================================
# 2. Movie features
# ======================================
def parse_genres(x):
    try:
        return [g["name"] for g in eval(x)]
    except:
        return []

movies["genres_list"] = movies["genres"].apply(parse_genres)
movies["overview"] = movies["overview"].fillna("")
movies["tagline"] = movies["tagline"].fillna("")
movies["text"] = movies["overview"] + " " + movies["tagline"]

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2)
)
X_text = tfidf.fit_transform(movies["text"])

mlb = MultiLabelBinarizer()
X_genres = csr_matrix(mlb.fit_transform(movies["genres_list"]))

num_cols = ["popularity", "runtime", "vote_average", "vote_count"]
movies[num_cols] = movies[num_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

scaler = StandardScaler()
X_num = csr_matrix(scaler.fit_transform(movies[num_cols]))

X_item_raw = hstack([X_text, X_genres, X_num])

# ======================================
# 3. Item embeddings
# ======================================
svd = TruncatedSVD(n_components=128, random_state=42)
X_item = svd.fit_transform(X_item_raw)

movie_id_to_idx = dict(zip(movies["id"], range(len(movies))))

# ======================================
# 4. Ratings → indices
# ======================================
ratings = ratings.copy()
ratings["movie_idx"] = ratings["movieId"].map(movie_id_to_idx)
ratings = ratings.dropna(subset=["movie_idx"])
ratings["movie_idx"] = ratings["movie_idx"].astype(int)

# ======================================
# 5. User embeddings (weighted mean)
# ======================================
user_embeddings = {}
for user_id, grp in ratings.groupby("userId"):
    idxs = grp["movie_idx"].values
    weights = grp["rating"].values
    user_embeddings[user_id] = np.average(
        X_item[idxs], axis=0, weights=weights
    )

user_ids = np.array(list(user_embeddings.keys()))
X_user = np.vstack(list(user_embeddings.values()))
user_id_to_idx = dict(zip(user_ids, range(len(user_ids))))

# ======================================
# 6. Train user-item classifier
# ======================================
model = SGDClassifier(
    loss="log_loss",
    learning_rate="optimal"
)

batch_size = 50000
ratings_list = list(ratings.itertuples(index=False))
n_batches = int(np.ceil(len(ratings_list) / batch_size))

for epoch in range(5):
    for i in range(n_batches):
        batch = ratings_list[i*batch_size:(i+1)*batch_size]

        X_batch, y_batch = [], []
        for row in batch:
            u, m = row.userId, row.movie_idx
            if u not in user_embeddings:
                continue

            X_batch.append(
                np.hstack([user_embeddings[u], X_item[m]])
            )
            y_batch.append(1 if row.rating >= 4.0 else 0)

        if X_batch:
            X_batch = np.array(X_batch, dtype=np.float32)
            y_batch = np.array(y_batch)

            if epoch == 0 and i == 0:
                model.partial_fit(X_batch, y_batch, classes=[0,1])
            else:
                model.partial_fit(X_batch, y_batch)

    print(f"Epoch {epoch+1} finished")

# ======================================
# 7. Recommendation logic
# ======================================
def recommend_for_user(user_id, top_n=20, alpha=0.7, beta=0.3):
    if user_id not in user_embeddings:
        return None

    user_vec = user_embeddings[user_id].reshape(1,-1)
    user_block = np.repeat(user_vec, X_item.shape[0], axis=0)
    X_pred = np.hstack([user_block, X_item])

    clf_score = model.predict_proba(X_pred)[:,1]
    emb_score = cosine_similarity(user_vec, X_item)[0]

    pop = movies["popularity"].values
    pop_penalty = 1 / (1 + np.log1p(pop))

    final_score = (
        alpha * clf_score +
        beta * emb_score
    ) * pop_penalty

    seen = ratings[ratings.userId == user_id]["movieId"].values

    recs = movies.copy()
    recs["score"] = final_score
    recs = recs[~recs["id"].isin(seen)]

    return recs.sort_values(
        "score", ascending=False
    ).head(top_n)[["title","genres_list","score"]]

# ======================================
# 8. Similar users
# ======================================
def similar_users(user_id, top_n=5):
    idx = user_id_to_idx[user_id]
    sims = cosine_similarity(
        X_user[idx].reshape(1,-1), X_user
    )[0]

    top_idx = np.argsort(sims)[::-1][1:top_n+1]
    return list(zip(user_ids[top_idx], sims[top_idx]))

# ======================================
# 9. LLM RERANK PROMPT
# ======================================
def build_llm_rerank_prompt(user_id, candidates):
    sims = similar_users(user_id, top_n=3)

    prompt = f"""
You are a professional movie recommender system.

User ID: {user_id}

Similar users:
"""
    for u, s in sims:
        prompt += f"- user {u}, similarity {s:.2f}\n"

    prompt += "\nCandidate movies:\n"
    for _, r in candidates.iterrows():
        prompt += (
            f"- {r['title']} | "
            f"genres: {r['genres_list']} | "
            f"score: {r['score']:.3f}\n"
        )

    prompt += """
Task:
1. Re-rank movies from best to worst
2. Explain the TOP 3 choices
3. Focus on latent preferences and diversity
4. Avoid generic explanations
"""

    return prompt

# ======================================
# 10. Run for 5 users
# ======================================
sample_users = np.random.choice(
    list(user_embeddings.keys()),
    size=5,
    replace=False
)

for uid in sample_users:
    print("\n" + "="*80)
    print(f"USER {uid}")
    print("="*80)

    candidates = recommend_for_user(uid, top_n=15)

    print("\n--- CANDIDATE MOVIES ---")
    print(candidates.head(5))

    print("\n--- SIMILAR USERS ---")
    print(similar_users(uid, top_n=3))

    print("\n--- LLM RERANK PROMPT ---")
    print(build_llm_rerank_prompt(uid, candidates))


Epoch 1 finished
Epoch 2 finished
Epoch 3 finished
Epoch 4 finished
Epoch 5 finished

USER 188079

--- CANDIDATE MOVIES ---
                        title                                   genres_list  \
5552   Grave of the Fireflies                       [Animation, Drama, War]   
40297          A Simple Story                                            []   
35903         Poil de Carotte                               [Drama, Comedy]   
19475               The Crush                              [Romance, Drama]   
27750                     Pan  [Action, Adventure, Fantasy, Drama, Romance]   

          score  
5552   0.529801  
40297  0.523800  
35903  0.523210  
19475  0.517230  
27750  0.502364  

--- SIMILAR USERS ---
[(220653, 0.9984860272292944), (224453, 0.9984806292785569), (106051, 0.998371622892904)]

--- LLM RERANK PROMPT ---

You are a professional movie recommender system.

User ID: 188079

Similar users:
- user 220653, similarity 1.00
- user 224453, similarity 1.00
- user 1